# NB2 — Small-File Problem & OPTIMIZE + ZORDER

**Mục tiêu:** prove the 3–10× speedup claim from slide §5.
Maps to deliverable bullet 2.

In [1]:
import sys, time, random
sys.path.append("/workspace/scripts")
from spark_session import get_spark
from delta.tables import DeltaTable

spark = get_spark("nb2_optimize_zorder")
path = "s3a://lakehouse/events_smallfiles"

## 0. Reset path (idempotent re-run)

Each run starts fresh — otherwise repeated appends keep growing the table
and the benchmark drifts.

In [2]:
spark.sql(f"DROP TABLE IF EXISTS delta.`{path}`")
# Best-effort: the DROP above unregisters the catalog entry, but Delta files
# may persist in MinIO. Overwrite below resets the data.

DataFrame[]

## 1. Manufacture the small-file problem

Append 200 tiny batches → 200 small files. Realistic streaming-ingestion shape.

In [3]:
for batch in range(200):
    rows = [(i, random.choice(["click", "view", "scroll", "purchase"]),
             random.randint(1, 10000))
            for i in range(batch * 500, (batch + 1) * 500)]
    df = spark.createDataFrame(rows, ["event_id", "kind", "user_id"])
    mode = "overwrite" if batch == 0 else "append"
    df.write.format("delta").mode(mode).save(path)

## 2. Benchmark BEFORE optimize

In [4]:
def bench(label):
    # Warm-up read so we measure query, not cold metadata fetch
    spark.read.format("delta").load(path).limit(1).count()
    t0 = time.time()
    n = (spark.read.format("delta").load(path)
            .where("user_id = 4242 AND kind = 'purchase'").count())
    dt = time.time() - t0
    print(f"{label:25s}  count={n}  time={dt:.2f}s")
    return dt

before = bench("BEFORE OPTIMIZE+ZORDER")

BEFORE OPTIMIZE+ZORDER     count=3  time=5.95s


In [5]:
spark.sql(f"DESCRIBE DETAIL delta.`{path}`").select(
    "numFiles", "sizeInBytes"
).show()


+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|    2400|    3449025|
+--------+-----------+



## 3. OPTIMIZE + ZORDER

In [6]:
spark.sql(f"OPTIMIZE delta.`{path}` ZORDER BY (user_id)")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,totalClusterPar

## 4. Benchmark AFTER

In [7]:
after = bench("AFTER OPTIMIZE+ZORDER")
speedup = before / max(after, 1e-6)
print(f"\nSpeedup: {speedup:.1f}×  (target ≥ 3×)")

AFTER OPTIMIZE+ZORDER      count=3  time=0.33s

Speedup: 18.1×  (target ≥ 3×)


## 5. Inspect file count change

In [8]:
spark.sql(f"DESCRIBE DETAIL delta.`{path}`").select(
    "numFiles", "sizeInBytes"
).show()

+--------+-----------+
|numFiles|sizeInBytes|
+--------+-----------+
|       1|     645588|
+--------+-----------+



## ✅ Deliverable check
- [ ] Speedup ≥ 3×
- [ ] `numFiles` dropped substantially after OPTIMIZE
- [ ] Screenshot the printed comparison

In [9]:
spark.stop()